In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import urllib.request
from PIL import Image
import numpy as np

# Image hosted on GitHub (PyTorch Hub repo)
url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"

# Download the image
******************************

# Load image
img = Image.open("dog.jpg").convert("RGB")
img = np.array(img)

# Convert to float32 in [0, 1]
img_float = tf.image.convert_image_dtype(img, tf.float32)

# Display image
plt.imshow(img_float)
plt.axis("off")
plt.title("Dog - PyTorch GitHub Sample")
plt.show()

In [ ]:
import tensorflow as tf

******************************

patch_size = 16

patches = tf.image.extract_patches(
    images=tf.expand_dims(image, axis=0),
******************************
    strides=[1, patch_size, patch_size, 1],
    rates=[1, 1, 1, 1],
    padding="VALID"
)

print("Patches shape:", patches.shape)

In [ ]:
num_patches_h = 224 // patch_size
num_patches_w = 224 // patch_size
******************************
******************************

patches_reshaped = tf.reshape(
    patches,
    (1, num_patches, patch_dim)
)

print("Number of patches:", num_patches)
print("Patch vector length:", patch_dim)
print("Patch sequence shape:", patches_reshaped.shape)

In [ ]:
patches_for_display = tf.reshape(
    patches,
    (num_patches_h, num_patches_w, patch_size, patch_size, 3)
)

plt.figure(figsize=(8, 8))

for i in range(num_patches):
    plt.subplot(num_patches_h, num_patches_w, i + 1)
    row = i // num_patches_w
    col = i % num_patches_w
    plt.imshow(patches_for_display[row, col])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
projection_dim = 64

******************************

patch_embeddings = projection(patches_reshaped)

print("Patch embeddings shape:", patch_embeddings.shape)

In [ ]:
position_embedding = tf.keras.layers.Embedding(
    input_dim=num_patches,
    output_dim=projection_dim
)

positions = tf.range(start=0, limit=num_patches, delta=1)

******************************

print("Encoded patches shape:", encoded_patches.shape)

In [ ]:
attention_layer = tf.keras.layers.MultiHeadAttention(
    num_heads=4,
    key_dim=projection_dim
)

******************************


print("Attention output shape:", attention_output.shape)

In [ ]:
def transformer_encoder(inputs, num_heads, projection_dim, mlp_dim):
    # Self-attention block
    x1 = tf.keras.layers.LayerNormalization()(inputs)
    attention_output = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=projection_dim
    )(x1, x1)

    # Residual connection
    x2 = tf.keras.layers.Add()([attention_output, inputs])

    # MLP block
******************************
    x3 = tf.keras.layers.Dense(mlp_dim, activation="gelu")(x3)
    x3 = tf.keras.layers.Dense(projection_dim)(x3)

    # Residual connection
    outputs = tf.keras.layers.Add()([x3, x2])

    return outputs

In [ ]:
x = encoded_patches

encoder_output = transformer_encoder(
    inputs=x,
    num_heads=4,
    projection_dim=64,
    mlp_dim=128
)

print("Input shape:", x.shape)
print("Encoder output shape:", encoder_output.shape)

In [ ]:
mlp = tf.keras.Sequential([
******************************
    tf.keras.layers.Dense(64)
])

mlp_output = mlp(encoded_patches)

print(mlp_output.shape)

In [ ]:
class Patches(tf.keras.layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
******************************
        )

        patch_dims = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [batch_size, -1, patch_dims]
        )

        return patches

In [ ]:
class PatchEncoder(tf.keras.layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()

        self.num_patches = num_patches
        self.projection = tf.keras.layers.Dense(projection_dim)
        self.position_embedding = tf.keras.layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patches):
******************************

        encoded = self.projection(patches) + self.position_embedding(positions)

        return encoded

In [ ]:
def create_vit_classifier(
    input_shape=(32, 32, 3),
    patch_size=4,
    num_classes=10,
    projection_dim=64,
    transformer_layers=4,
    num_heads=4,
    mlp_dim=128):
    inputs = tf.keras.Input(shape=input_shape)

    patches = Patches(patch_size)(inputs)

    num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)

    encoded_patches = PatchEncoder(
        num_patches=num_patches,
        projection_dim=projection_dim)(patches)

    x = encoded_patches

    for _ in range(transformer_layers):
        x = transformer_encoder(
            x,
            num_heads=num_heads,
            projection_dim=projection_dim,
            mlp_dim=mlp_dim)

    x = tf.keras.layers.LayerNormalization()(x)
******************************
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    return model

In [ ]:
vit_model = create_vit_classifier(
    input_shape=(32, 32, 3),
    patch_size=4,
    num_classes=10
)

vit_model.summary()

In [ ]:
vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10

# Load CIFAR-10
******************************

# Normalize images to [0, 1]
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten labels: (N, 1) -> (N,)
y_train_full = y_train_full.squeeze()
y_test = y_test.squeeze()

# CIFAR-10 has 50,000 training images
# Use 80% train, 20% validation
split = int(0.8 * len(x_train_full))

x_train = x_train_full[:split]
y_train = y_train_full[:split]

x_val = x_train_full[split:]
y_val = y_train_full[split:]

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

In [ ]:
history_vit = vit_model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64
)

In [ ]:
test_loss, test_acc = vit_model.evaluate(x_test, y_test)

print("ViT test loss:", test_loss)
print("ViT test accuracy:", test_acc)

In [ ]:
batch_size = tf.shape(encoded_patches)[0]
projection_dim = encoded_patches.shape[-1]

cls_token = tf.Variable(
    tf.zeros((1, 1, projection_dim)),
    trainable=True
)

cls_tokens = tf.tile(cls_token, [batch_size, 1, 1])

******************************


print("Before CLS:", encoded_patches.shape)
print("After CLS:", tokens_with_cls.shape)

In [ ]:
!pip install -q keras-hub

In [ ]:
import keras
import keras_hub
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request

preset = "vit_base_patch16_224_imagenet"

vit = keras_hub.models.ImageClassifier.from_preset(preset)

vit.summary()

In [ ]:
url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
urllib.request.urlretrieve(url, "dog.jpg")

img = Image.open("dog.jpg").convert("RGB")
img = np.array(img)

plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(preset)

img_processed = preprocessor(img)

print("Processed shape:", img_processed.shape)

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications.imagenet_utils import decode_predictions

# Make prediction
predictions = vit.predict(np.expand_dims(img, axis=0))

# Convert raw scores to probabilities
******************************

# Get predicted class
class_index = np.argmax(probs)

# Map ImageNet class index to class name
class_name = decode_predictions(
    np.expand_dims(probs, axis=0),
    top=1
)[0][0][1]

# Display results
print("Prediction shape:", predictions.shape)
print("Predicted class index:", class_index)
print("Predicted class name:", class_name)
print("Probability:", probs[class_index])